# Part 2: RFM Segmentation & Retention Strategy
**Objective:** Build RFM (Recency, Frequency, Monetary) features, combine them with non-RFM behavioral signals, and segment the customer base to prioritize retention actions.

# **Name: Donepudi Chalapathi Kalyan**
# **ID: iitp_aiml_2506015**

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

data_path = 'data/d2c_churn_data_package/'


### 1. Constructing RFM Features
We will compute Recency, Frequency, and Monetary value for each customer using only pre-snapshot orders.

In [2]:
# Load and clean orders
orders = pd.read_csv(data_path + 'orders.csv')
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Filter out post-snapshot leakage data
snapshot_date = pd.to_datetime('2025-09-30')
orders_pre = orders[orders['order_date'] <= snapshot_date].copy()

# Deduplicate
orders_pre['order_id_clean'] = orders_pre['order_id'].str.replace('_DUP', '')
orders_clean = orders_pre.drop_duplicates(subset=['order_id_clean', 'category', 'quantity', 'gross_amount', 'discount_pct', 'delivery_days', 'returned', 'rating'])

# Calculate RFM per customer
rfm = orders_clean.groupby('customer_id').agg(
    max_order_date=('order_date', 'max'),
    frequency=('order_id_clean', 'nunique'),
    monetary=('gross_amount', 'sum'),
    avg_discount=('discount_pct', 'mean')
).reset_index()

rfm['recency'] = (snapshot_date - rfm['max_order_date']).dt.days
rfm.drop(columns=['max_order_date'], inplace=True)
print(rfm.head())

  customer_id  frequency  monetary  avg_discount  recency
0   CUST00001          6   2955.57      0.363333      107
1   CUST00002          1    581.00      0.230000       40
2   CUST00003          1    649.98      0.470000      171
3   CUST00004          1   1604.04      0.160000      131
4   CUST00005          4   2550.91      0.442500       38


### 2. Adding Non-RFM Signals
We integrate support ticket counts and web activity (days since last visit) to add behavioral context.

In [3]:
# Support Tickets
tickets = pd.read_csv(data_path + 'support_tickets.csv')
cust_tickets = tickets.groupby('customer_id').agg(
    support_ticket_count=('ticket_id', 'count')
).reset_index()

# Web Events
web = pd.read_csv(data_path + 'web_events_snapshot.csv')

# Merge
df = rfm.merge(cust_tickets, on='customer_id', how='left')
df = df.merge(web[['customer_id', 'last_visit_days_ago']], on='customer_id', how='left')

# Impute missing
df['support_ticket_count'] = df['support_ticket_count'].fillna(0)
df['last_visit_days_ago'] = df['last_visit_days_ago'].fillna(df['last_visit_days_ago'].max() + 30)

print(f"Data shape after adding non-RFM signals: {df.shape}")

Data shape after adding non-RFM signals: (2400, 7)


### 3. Customer Segmentation Logic
We classify customers into 7 distinct segments based on RFM and behavioral signals.

In [4]:
def assign_segment(row):
    r, f, m = row['recency'], row['frequency'], row['monetary']
    tickets = row['support_ticket_count']
    visit = row['last_visit_days_ago']
    discount = row['avg_discount']
    
    # 1. High-Value but Unhappy
    if m >= 1000 and tickets >= 2:
        return 'High-Value but Unhappy'
    # 2. Dormant Customers
    elif r > 120:
        return 'Dormant Customers'
    # 3. Discount-Sensitive Customers
    elif discount > 0.3:
        return 'Discount-Sensitive Customers'
    # 4. Low-Engagement Customers
    elif visit > 20 and f == 1:
        return 'Low-Engagement Customers'
    # 5. Champions
    elif r <= 30 and f >= 3 and m > 1500:
        return 'Champions'
    # 6. Loyal Customers
    elif r <= 60 and f >= 2:
        return 'Loyal Customers'
    # 7. At-Risk Customers
    else:
        return 'At-Risk Customers'

df['segment_name'] = df.apply(assign_segment, axis=1)

print("Segment Distribution:")
print(df['segment_name'].value_counts())

Segment Distribution:
segment_name
Dormant Customers               559
Discount-Sensitive Customers    558
High-Value but Unhappy          457
At-Risk Customers               452
Loyal Customers                 186
Champions                       145
Low-Engagement Customers         43
Name: count, dtype: int64


### 4. Export to CSV
Saving the results for downstream retention campaigns.

In [5]:
df.to_csv('segments.csv', index=False)
print("Saved 2400 customer segments to 'segments.csv'.")

Saved 2400 customer segments to 'segments.csv'.
